# Evaluating samplers on the SPD manifold

This notebook implements two heat-kernel-based targets on the manifold of symmetric positive definite matrices `SPD(n)` under the affine-invariant metric:

1. **SMS short-time kernel**: a Varadhan / Van-Vleck-Morette corrected asymptotic kernel.
2. **Borovitskiy-style exact `n=2` kernel**: an exact product formula for `SPD(2) \cong \mathbb{R} \times \mathbb{H}^2_{K^2=1/2}`.  
   For `n>2`, the notebook falls back to the SMS kernel unless you later replace that block by a full Harish–Chandra / Plancherel numerical implementation.

It also includes:

- rejection sampling and importance sampling from a log-Euclidean Gaussian proposal,
- four experiments:
  - scaling with dimension,
  - small-time / Varadhan stability,
  - Monte Carlo integration error,
  - wall-clock tradeoff.

## Important scope note

This notebook is rigorous where the formulas are standard and explicitly implemented:

- affine-invariant metric on `SPD(n)`,
- Van Vleck correction used in the short-time kernel,
- exact `SPD(2)` product decomposition kernel.

For **general `n \ge 3`**, this notebook **does not implement** the full exact spectral heat kernel on `GL(n)/O(n)`. Instead, it uses the SMS kernel as the practical target. That is deliberate: a full Harish–Chandra numerical implementation is substantially heavier and would deserve a separate notebook.

In [ ]:
import math
import time
import itertools
from dataclasses import dataclass

import numpy as np
import numpy.linalg as npl
import scipy.linalg as spla
import scipy.integrate as integrate
import matplotlib.pyplot as plt

rng = np.random.default_rng(12345)
plt.rcParams["figure.figsize"] = (7, 4.5)

## 1. SPD utilities

We work with the affine-invariant metric
$$
g_X(U,V) = \operatorname{tr}(X^{-1} U X^{-1} V).
$$

If $X,Y \in \mathrm{SPD}(n)$, then
$$
d_{\mathrm{AI}}(X,Y)^2
= \left\| \log\!\left(X^{-1/2} Y X^{-1/2}\right) \right\|_F^2.
$$

For the short-time kernel, we also use the standard Jacobian / Van-Vleck factor in Cartan coordinates:
$$
J(a) = \prod_{i<j} \frac{\sinh\!\big((a_i-a_j)/2\big)}{(a_i-a_j)/2},
\qquad
\Delta_{\mathrm{VVM}}^{1/2}(a) = J(a)^{-1/2}.
$$
Equivalently,
$$
\log \Delta_{\mathrm{VVM}}^{1/2}(a)
=
\frac12 \sum_{i<j}
\log\!\left(
\frac{|a_i-a_j|}{2\sinh(|a_i-a_j|/2)}
\right).
$$


In [ ]:
def sym(A):
    return 0.5 * (A + A.T)

def is_spd(X, tol=1e-12):
    if not np.allclose(X, X.T, atol=1e-12):
        return False
    eigs = npl.eigvalsh(X)
    return np.min(eigs) > tol

def spd_dim(n):
    return n * (n + 1) // 2

def sqrtm_spd(X):
    w, V = npl.eigh(X)
    return (V * np.sqrt(np.clip(w, 1e-15, None))) @ V.T

def invsqrtm_spd(X):
    w, V = npl.eigh(X)
    return (V * (1.0 / np.sqrt(np.clip(w, 1e-15, None)))) @ V.T

def logm_spd(X):
    w, V = npl.eigh(X)
    return (V * np.log(np.clip(w, 1e-300, None))) @ V.T

def expm_sym(A):
    w, V = npl.eigh(sym(A))
    return (V * np.exp(w)) @ V.T

def ai_log_spectrum(X, Y):
    Xi2 = invsqrtm_spd(X)
    Z = Xi2 @ Y @ Xi2
    vals = np.clip(npl.eigvalsh(sym(Z)), 1e-300, None)
    a = np.log(vals)
    return np.sort(a)

def ai_distance2(X, Y):
    a = ai_log_spectrum(X, Y)
    return float(np.dot(a, a))

def ai_distance(X, Y):
    return math.sqrt(ai_distance2(X, Y))

def loge_distance2(X, Y):
    LX = logm_spd(X)
    LY = logm_spd(Y)
    D = LX - LY
    return float(np.sum(D * D))

def _log_sinh_over_x_small(z):
    # Returns log(sinh(z)/z) stably for small z.
    az = abs(z)
    if az < 1e-6:
        z2 = z * z
        return z2 / 6.0 - z2 * z2 / 180.0
    return math.log(math.sinh(z) / z)

def log_vvm_sqrt_from_log_eigs(a):
    # a: sorted log-eigenvalues
    n = len(a)
    total = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            d = abs(a[j] - a[i])
            if d < 1e-10:
                continue
            total += 0.5 * (math.log(d) - math.log(2.0) - math.log(math.sinh(d / 2.0)))
    return total

def log_vvm_sqrt(X, Y):
    a = ai_log_spectrum(X, Y)
    return log_vvm_sqrt_from_log_eigs(a)

## 2. Targets

### 2.1 SMS short-time kernel

We use the standard short-time form
$$
p_t^{\mathrm{SMS}}(X,Y)
\approx
(4\pi t)^{-m/2}
\exp\!\left(
-\frac{d_{\mathrm{AI}}(X,Y)^2}{4t}
\right)
\Delta_{\mathrm{VVM}}^{1/2}(X,Y),
$$
where $m = \dim(\mathrm{SPD}(n)) = n(n+1)/2\$.

This is the correct local heat-kernel asymptotic to first order in time. It is not the full exact kernel for general `n`.

### 2.2 Exact `SPD(2)` product kernel

For `SPD(2)` with the affine-invariant metric,
$$
\mathrm{SPD}(2) \cong \mathbb{R} \times \mathbb{H}^2_{K^2=1/2}
$$
as a Riemannian product.

Let $a_1,a_2\$ be the log-eigenvalues of $X^{-1/2} Y X^{-1/2}$, and define
$$
u = \frac{a_1+a_2}{\sqrt{2}}, \qquad
r = \frac{|a_1-a_2|}{\sqrt{2}}.
$$
Then the exact heat kernel factors as
$$
p_t^{\mathrm{SPD}(2)}(X,Y)
=
p_t^{\mathbb{R}}(u)\,
p_t^{\mathbb{H}^2_{K^2=1/2}}(r).
$$

For the hyperbolic factor with curvature \(-K^2\), scaling gives
$$
p_t^{\mathbb{H}^2_{K^2}}(r)
=
K^2\, p_{K^2 t}^{\mathbb{H}^2_1}(Kr),
$$
where $p_s^{\mathbb{H}^2_1}$ is McKean's formula on the curvature $-1$ hyperbolic plane.


In [ ]:
def log_sms_kernel(X, Y, t):
    n = X.shape[0]
    m = spd_dim(n)
    d2 = ai_distance2(X, Y)
    lvvm = log_vvm_sqrt(X, Y)
    return -0.5 * m * math.log(4.0 * math.pi * t) - d2 / (4.0 * t) + lvvm

def h2_heat_kernel_curv_minus1(r, t):
    # McKean formula on H^2 with curvature -1.
    # Numerically stable enough for moderate t and r.
    if t <= 0:
        raise ValueError("t must be positive")
    if r < 0:
        raise ValueError("r must be nonnegative")

    def integrand(s):
        denom = math.sqrt(max(math.cosh(s) - math.cosh(r), 1e-300))
        return s * math.exp(-s * s / (4.0 * t)) / denom

    upper = max(r + 12.0 * math.sqrt(t) + 8.0, r + 20.0)
    val, err = integrate.quad(integrand, r, upper, epsabs=1e-10, epsrel=1e-8, limit=400)
    pref = math.exp(-t / 4.0) / ((4.0 * math.pi * t) ** 1.5 * math.sqrt(2.0))
    return pref * val

def h2_heat_kernel_curv_minus_K2(r, t, K):
    return (K ** 2) * h2_heat_kernel_curv_minus1(K * r, (K ** 2) * t)

def log_spd2_exact_kernel(X, Y, t):
    a = ai_log_spectrum(X, Y)
    a1, a2 = float(a[0]), float(a[1])
    u = (a1 + a2) / math.sqrt(2.0)
    r = abs(a1 - a2) / math.sqrt(2.0)

    p_R = (4.0 * math.pi * t) ** (-0.5) * math.exp(-(u * u) / (4.0 * t))
    K = 1.0 / math.sqrt(2.0)  # curvature -1/2
    p_H = h2_heat_kernel_curv_minus_K2(r, t, K)
    return math.log(max(p_R * p_H, 1e-300))

def log_target_kernel(X, Y, t, method="sms"):
    n = X.shape[0]
    if method == "sms":
        return log_sms_kernel(X, Y, t)
    elif method == "borovitskiy":
        if n == 2:
            return log_spd2_exact_kernel(X, Y, t)
        # Honest fallback:
        return log_sms_kernel(X, Y, t)
    else:
        raise ValueError("Unknown method")

## 3. Proposal and samplers

We use a **log-Euclidean Gaussian proposal** centered at $X_0$:
1. sample a symmetric Gaussian matrix $H$ in the tangent / log coordinates,
2. set
$$
Y = X_0^{1/2} \exp(H) X_0^{1/2}.
$$

The proposal density in log-coordinates is the Gaussian density in the symmetric matrix space. This is not the exact Riemannian normal proposal, but it is a reasonable practical proposal for both rejection and importance sampling.


In [ ]:
def sample_sym_gaussian(n, sigma, rng=rng):
    A = rng.normal(size=(n, n))
    H = np.triu(A)
    H = H + H.T - np.diag(np.diag(H))
    H *= sigma / math.sqrt(2.0)
    H[np.diag_indices(n)] *= math.sqrt(2.0)
    return sym(H)

def sample_logeuclidean_proposal(X0, t, rng=rng):
    n = X0.shape[0]
    sigma = math.sqrt(2.0 * t)
    H = sample_sym_gaussian(n, sigma=sigma, rng=rng)
    Xh = sqrtm_spd(X0)
    Y = Xh @ expm_sym(H) @ Xh
    return sym(Y), H

def log_logeuclidean_proposal_density(H, t):
    # Density in Sym(n) coordinates for H ~ N(0, 2t I) on independent coordinates.
    n = H.shape[0]
    m = spd_dim(n)
    fro2 = float(np.sum(H * H))
    return -0.5 * m * math.log(4.0 * math.pi * t) - fro2 / (4.0 * t)

def get_H_from_sample(X0, Y):
    Xh_inv = invsqrtm_spd(X0)
    Z = Xh_inv @ Y @ Xh_inv
    return logm_spd(Z)

def importance_sampler(X0, t, N, method="sms", rng=rng):
    samples = []
    logw = np.empty(N, dtype=float)

    for k in range(N):
        Y, H = sample_logeuclidean_proposal(X0, t, rng=rng)
        lt = log_target_kernel(X0, Y, t, method=method)
        lq = log_logeuclidean_proposal_density(H, t)
        logw[k] = lt - lq
        samples.append(Y)

    c = np.max(logw)
    w = np.exp(logw - c)
    w_sum = np.sum(w)
    wn = w / w_sum
    ess = (w_sum ** 2) / np.sum(w ** 2)
    return {
        "samples": samples,
        "logw": logw,
        "weights": wn,
        "ess": ess,
    }

def calibrate_rejection_envelope(X0, t, method="sms", n_cal=500, buffer=0.5, rng=rng):
    vals = []
    for _ in range(n_cal):
        Y, H = sample_logeuclidean_proposal(X0, t, rng=rng)
        lt = log_target_kernel(X0, Y, t, method=method)
        lq = log_logeuclidean_proposal_density(H, t)
        vals.append(lt - lq)
    return max(vals) + buffer

def rejection_sampler(X0, t, N, method="sms", logM=None, n_cal=500, rng=rng):
    if logM is None:
        logM = calibrate_rejection_envelope(X0, t, method=method, n_cal=n_cal, rng=rng)

    samples = []
    trials = 0
    while len(samples) < N:
        Y, H = sample_logeuclidean_proposal(X0, t, rng=rng)
        lt = log_target_kernel(X0, Y, t, method=method)
        lq = log_logeuclidean_proposal_density(H, t)
        if math.log(rng.uniform()) <= lt - lq - logM:
            samples.append(Y)
        trials += 1

    return {
        "samples": samples,
        "trials": trials,
        "acceptance_rate": N / trials,
        "logM": logM,
    }

## 4. Basic sanity checks

In [ ]:
I2 = np.eye(2)
Y = np.array([[2.0, 0.3],[0.3, 1.2]])
print("SPD check:", is_spd(Y))
print("AI distance^2(I,Y):", ai_distance2(I2, Y))
print("log SMS kernel:", log_sms_kernel(I2, Y, t=0.4))
print("log exact SPD(2) kernel:", log_spd2_exact_kernel(I2, Y, t=0.4))

## 5. Experiment 1: dimension scaling

This checks:

- rejection acceptance rate versus dimension,
- importance-sampling ESS versus dimension.

For `method="borovitskiy"`, only `n=2` is exact in this notebook. For `n>2`, the code honestly falls back to the SMS target.


In [ ]:
def random_spd_identity_center(n):
    return np.eye(n)

def experiment_dimension_scaling(dims=(2,3,5,8), t=0.2, N=300, rng=rng):
    rows = []
    for n in dims:
        X0 = np.eye(n)

        t0 = time.perf_counter()
        rej = rejection_sampler(X0, t=t, N=N, method="sms", n_cal=300, rng=rng)
        t1 = time.perf_counter()

        imp = importance_sampler(X0, t=t, N=N, method="sms", rng=rng)
        t2 = time.perf_counter()

        rows.append({
            "n": n,
            "dim_manifold": spd_dim(n),
            "rej_accept": rej["acceptance_rate"],
            "rej_time_sec": t1 - t0,
            "imp_ess": imp["ess"],
            "imp_ess_over_N": imp["ess"] / N,
            "imp_time_sec": t2 - t1,
        })
    return rows

rows = experiment_dimension_scaling()
rows

In [ ]:
dims = [r["n"] for r in rows]
acc = [r["rej_accept"] for r in rows]
essr = [r["imp_ess_over_N"] for r in rows]

plt.figure()
plt.plot(dims, acc, marker="o")
plt.xlabel("matrix dimension n")
plt.ylabel("rejection acceptance rate")
plt.title("Experiment 1: rejection acceptance vs n")
plt.show()

plt.figure()
plt.plot(dims, essr, marker="o")
plt.xlabel("matrix dimension n")
plt.ylabel("ESS / N")
plt.title("Experiment 1: importance-sampling ESS ratio vs n")
plt.show()

## 6. Experiment 2: small-time / Varadhan stability

We fix `n` and drive \(t \downarrow 0\). The main quantity is the variance of the importance weights. A good proposal should keep normalized weight degeneracy under control as long as possible.


In [ ]:
def experiment_small_time(n=2, ts=(0.5, 0.2, 0.1, 0.05, 0.02), N=500, method="borovitskiy", rng=rng):
    X0 = np.eye(n)
    rows = []
    for t in ts:
        out = importance_sampler(X0, t=t, N=N, method=method, rng=rng)
        logw = out["logw"]
        lw = logw - np.max(logw)
        w = np.exp(lw)
        w /= np.sum(w)
        rows.append({
            "t": t,
            "ess": out["ess"],
            "ess_over_N": out["ess"] / N,
            "weight_var": float(np.var(w)),
            "max_weight": float(np.max(w)),
        })
    return rows

rows_small = experiment_small_time()
rows_small

In [ ]:
ts = [r["t"] for r in rows_small]
essr = [r["ess_over_N"] for r in rows_small]
mwx = [r["max_weight"] for r in rows_small]

plt.figure()
plt.plot(ts, essr, marker="o")
plt.gca().invert_xaxis()
plt.xlabel("t")
plt.ylabel("ESS / N")
plt.title("Experiment 2: ESS ratio as t decreases")
plt.show()

plt.figure()
plt.plot(ts, mwx, marker="o")
plt.gca().invert_xaxis()
plt.xlabel("t")
plt.ylabel("max normalized weight")
plt.title("Experiment 2: largest normalized weight as t decreases")
plt.show()

## 7. Experiment 3: Monte Carlo integration error

We test
\[
f(Y)=d_{\mathrm{AI}}(I,Y)^2.
\]

For very small time, the heat semigroup gives the local asymptotic
\[
\mathbb{E}[f(Y)] \approx 2\, \dim(\mathrm{SPD}(n))\, t.
\]

That is a **small-time benchmark**, not a globally exact identity. So interpret this experiment as a local consistency check.


In [ ]:
def f_sqdist_I(Y):
    n = Y.shape[0]
    return ai_distance2(np.eye(n), Y)

def weighted_mean(samples, weights, f):
    vals = np.array([f(S) for S in samples], dtype=float)
    return float(np.sum(weights * vals))

def experiment_mc_error(n=2, t=0.05, Ns=(100, 200, 500, 1000), method="borovitskiy", rng=rng):
    X0 = np.eye(n)
    m = spd_dim(n)
    truth_local = 2.0 * m * t
    rows = []
    for N in Ns:
        out = importance_sampler(X0, t=t, N=N, method=method, rng=rng)
        est = weighted_mean(out["samples"], out["weights"], f_sqdist_I)
        err = abs(est - truth_local)
        rows.append({
            "N": N,
            "estimate": est,
            "local_truth": truth_local,
            "abs_error": err,
            "ess": out["ess"],
        })
    return rows

rows_mc = experiment_mc_error()
rows_mc

In [ ]:
Ns = [r["N"] for r in rows_mc]
errs = [r["abs_error"] for r in rows_mc]

plt.figure()
plt.loglog(Ns, errs, marker="o")
plt.xlabel("N")
plt.ylabel("absolute error")
plt.title("Experiment 3: MC error for E[d_AI(I,Y)^2]")
plt.show()

## 8. Experiment 4: wall-clock tradeoff

Here we compare the cost of evaluating:

- SMS kernel,
- exact `SPD(2)` kernel.

This is the clean comparison that this notebook can support without pretending to have a full higher-rank exact spectral implementation.


In [ ]:
def benchmark_kernel_eval(n=2, t=0.2, M=200, rng=rng):
    X0 = np.eye(n)
    Ys = [sample_logeuclidean_proposal(X0, t=t, rng=rng)[0] for _ in range(M)]

    t0 = time.perf_counter()
    vals_sms = [log_target_kernel(X0, Y, t, method="sms") for Y in Ys]
    t1 = time.perf_counter()

    vals_bor = [log_target_kernel(X0, Y, t, method="borovitskiy") for Y in Ys]
    t2 = time.perf_counter()

    return {
        "n": n,
        "M": M,
        "sms_time_sec": t1 - t0,
        "borovitskiy_time_sec": t2 - t1,
        "sms_avg_ms": 1000.0 * (t1 - t0) / M,
        "borovitskiy_avg_ms": 1000.0 * (t2 - t1) / M,
        "mean_abs_logdiff": float(np.mean(np.abs(np.array(vals_sms) - np.array(vals_bor)))),
    }

bench = benchmark_kernel_eval()
bench

## 9. Optional visualization in `SPD(2)`

We visualize sampled points in the two log-eigenvalue coordinates.


In [ ]:
def logeig_coords(X):
    a = np.log(np.clip(npl.eigvalsh(X), 1e-300, None))
    return np.sort(a)

out = importance_sampler(np.eye(2), t=0.15, N=300, method="borovitskiy", rng=rng)
pts = np.array([logeig_coords(S) for S in out["samples"]])
w = out["weights"]

plt.figure()
plt.scatter(pts[:,0], pts[:,1], s=8 + 300*w, alpha=0.7)
plt.xlabel("log eigenvalue 1")
plt.ylabel("log eigenvalue 2")
plt.title("Importance samples in log-eigenvalue coordinates")
plt.show()

## 10. Conclusions and honest limitations

What this notebook gives you:

- a correct affine-invariant geometry implementation,
- a practical short-time heat-kernel target for all `n`,
- an exact `SPD(2)` kernel via the `R \times H^2` decomposition,
- executable experiments for sampler diagnostics.

What it does **not** give you:

- a full exact higher-rank spectral heat kernel on `GL(n)/O(n)` for `n \ge 3`,
- a rigorous deterministic TV-distance computation against a closed-form exact target in higher rank.

If you want the next step, the natural extension is:
1. implement the Harish–Chandra / Plancherel integral for `SL(n)/SO(n)`,
2. combine it with the flat determinant direction,
3. replace the fallback block in `log_target_kernel`.
